# Patient-Friendly Medical Rewriter — QLoRA Training (Colab)

Fine-tunes `Qwen/Qwen2-1.5B-Instruct` with QLoRA on Cochrane medical
abstract -> plain-language pairs.

**Runtime:** set to GPU (T4 is enough). `Runtime -> Change runtime type -> T4 GPU`.

**Before running:** upload `train.jsonl` and `val.jsonl` (from your local
`data/processed/`) using the file panel on the left, or run the upload cell below.

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers peft bitsandbytes trl accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.4 MB/s eta 0:00:00


## 2. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Set Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


## 3. Upload training data

Run this and select your local `train.jsonl` and `val.jsonl`.
(Skip if you already uploaded them via the file panel.)

In [ ]:
import os
from google.colab import files

os.makedirs('data/processed', exist_ok=True)

if not (os.path.exists('data/processed/train.jsonl') and os.path.exists('data/processed/val.jsonl')):
    print('Select train.jsonl and val.jsonl...')
    uploaded = files.upload()
    for fname in uploaded:
        dest = os.path.join('data/processed', os.path.basename(fname))
        os.rename(fname, dest)
        print('Saved', dest)
else:
    print('Data already present.')

Select train.jsonl and val.jsonl...


Saving train.jsonl to train.jsonl
Saving val.jsonl to val.jsonl
Saved data/processed/train.jsonl
Saved data/processed/val.jsonl


In [ ]:
import json, random

def subsample(path, n=300, seed=42):
    with open(path) as f:
        lines = f.readlines()
    random.Random(seed).shuffle(lines)
    with open(path, 'w') as f:
        f.writelines(lines[:n])

subsample('data/processed/train.jsonl', n=300)
subsample('data/processed/val.jsonl', n=50)

## 4. Training code

Self-contained copy of the QLoRA logic from `scripts/model.py`, so the
notebook runs without uploading the whole repo. Keep this in sync with
`scripts/model.py` if you change hyperparameters.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import inspect

MODEL_ID = 'Qwen/Qwen2-1.5B-Instruct'
ADAPTER_DIR = 'models/qlora-adapter'


def _use_bf16():
    """Native bf16 needs Ampere (8.0)+. T4 (7.5) lacks it."""
    if not torch.cuda.is_available():
        return False
    major, _ = torch.cuda.get_device_capability()
    return major >= 8


def get_compute_dtype():
    # bf16 on Ampere+; on T4 use fp32 to avoid the fp16 GradScaler entirely
    # (fp16 GradScaler crashes on Qwen2's bf16 grads; T4 runs fp32 natively).
    return torch.bfloat16 if _use_bf16() else torch.float32


def get_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=get_compute_dtype(),
        bnb_4bit_use_double_quant=True,
    )


def get_lora_config():
    return LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        task_type='CAUSAL_LM',
    )


def train_model(num_epochs=1, batch_size=4, learning_rate=2e-4,
                max_seq_length=1536, output_dir=ADAPTER_DIR):
    """Fine-tune Qwen2-1.5B-Instruct with QLoRA (pure fp32 on T4)."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=get_bnb_config(),
        dtype=get_compute_dtype(),
        device_map='auto',
        trust_remote_code=True,
    )
    model.config.use_cache = False

    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, get_lora_config())

    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    model.print_trainable_parameters()
    dtypes = {p.dtype for p in model.parameters() if p.requires_grad}
    print('Trainable param dtypes:', dtypes)

    dataset = load_dataset('json', data_files={
        'train': 'data/processed/train.jsonl',
        'validation': 'data/processed/val.jsonl',
    })

    sft_params = inspect.signature(SFTConfig.__init__).parameters
    seq_len_key = 'max_length' if 'max_length' in sft_params else 'max_seq_length'

    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=4,
        learning_rate=learning_rate,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        logging_steps=25,
        eval_strategy='no',
        save_strategy='epoch',
        save_total_limit=2,
        bf16=_use_bf16(),
        fp16=False,
        report_to='none',
        seed=42,
        dataset_num_proc=1,
        **{seq_len_key: max_seq_length},
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset['train'],
        eval_dataset=dataset['validation'],
        processing_class=tokenizer,
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print('Adapter saved to', output_dir)

## 5. Run training

On a T4 with ~4k examples and 3 epochs this takes roughly 30-50 minutes.

In [ ]:
train_model()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Truncating eval dataset (num_proc=1):   0%|          | 0/50 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset (num_proc=1):   0%|          | 0/50 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss


Adapter saved to models/qlora-adapter


## 6. Quick before/after sanity check

In [9]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 7.4 MB/s eta 0:00:00


In [10]:

from peft import PeftModel

SYSTEM_PROMPT = (
    'You are a medical text simplifier. Rewrite the following medical text '
    'into plain language that a patient with no medical background can '
    'understand. Preserve all key findings and conclusions. Do not add '
    'information not present in the original text.'
)

TEST_TEXT = (
    'A meta-analysis of randomized controlled trials demonstrated a '
    'statistically significant reduction in glycated hemoglobin (HbA1c) '
    'levels (mean difference -0.5%, 95% CI -0.7 to -0.3, p<0.001) in '
    'patients receiving the intervention compared to placebo.'
)


def generate(model, tokenizer, text):
    """Generate a plain-language rewrite."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=True,
                             temperature=0.7, top_p=0.9, repetition_penalty=1.1)
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


tok = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
print('=== BEFORE (base) ===')
print(generate(base, tok, TEST_TEXT))

tuned = PeftModel.from_pretrained(base, ADAPTER_DIR)
print('\n=== AFTER (fine-tuned) ===')
print(generate(tuned, tok, TEST_TEXT))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== BEFORE (base) ===


There was a study that looked at people who got special treatment or took medicine. The study found that these people had lower numbers in their blood sugar (called HbA1c) than other people didn't. This means they were healthier! The change happened by an average of -0.5%. This difference is small but still shows a clear effect when looking at the whole group of people. They did this research because they wanted to know if taking certain medicines could help make people's blood sugar better.

=== AFTER (fine-tuned) ===
The study found that people who took a supplement for one year had a small decrease in their average blood sugar over time. This means that they were able to control their blood sugar better than those on the placebo. The researchers did not find any evidence that this effect was caused by the supplement itself or by anything else. However, it is important to note that the study only looked at whether taking a supplement would help people manage their blood sugar; it did

## 7. Download the trained adapter

Zips the adapter (only a few MB — LoRA weights are tiny) and downloads it.
Unpack into your local `models/qlora-adapter/`.

In [8]:
import shutil
from google.colab import files

shutil.make_archive('qlora-adapter', 'zip', ADAPTER_DIR)
files.download('qlora-adapter.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import peft
import transformers
import torch

print(peft.__version__)
print(transformers.__version__)
print(torch.__version__)

0.20.0
5.14.1
2.11.0+cu128


In [12]:
import transformers
print(transformers.__version__)

5.14.1


In [13]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_ID)
model = PeftModel.from_pretrained(base, "models/qlora-adapter")

merged = model.merge_and_unload()
merged.save_pretrained("merged_model")
tokenizer.save_pretrained("merged_model")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

NameError: name 'tokenizer' is not defined